# 00 - Setup: Infrastructure and Configuration

This notebook sets up the foundational infrastructure for the Healthcare ML Pipeline.

## What This Notebook Does

| Step | Description | Snowflake Services Used |
|------|-------------|------------------------|
| 1 | Import libraries and load config | Python, YAML |
| 2 | Create Snowpark session | Snowpark Session |
| 3 | Create database, schema, warehouse | DDL via Snowpark |
| 4 | Create tables for pipeline | DDL via Snowpark |
| 5 | Create stages for artifacts | Internal Stages |

## Imports and Configuration

Import required libraries and load the pipeline configuration from `config.yaml`.

In [ ]:
%cd ..

In [ ]:
import os
import sys
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from snowflake.snowpark import Session
from source.configs import get_config
from source.utils import get_session

config = get_config("source/config.yaml")

print(f"Database: {config.snowflake.database}")
print(f"Schema: {config.snowflake.schema_name}")
print(f"Warehouse: {config.snowflake.warehouse}")
print(f"Model Name: {config.model.model_name}")
print(f"Compute Pool: {config.compute.compute_pool}")

## Create Snowpark Session

Create a Snowpark session using a named connection from `~/.snowflake/connections.toml`.

In [ ]:
# session = get_session(config.snowflake.connection_name)
session = get_session("DEMO")

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")

---
## Create Database, Schema, and Warehouse

Create the foundational Snowflake objects required for the ML pipeline.

| Object | Purpose |
|--------|---------|
| **Database** | Container for all pipeline objects |
| **Schema** | Namespace for tables, views, models |
| **Warehouse** | Compute resources for queries and ML |

In [ ]:
from setup.database_setup import DatabaseSetup

db_setup = DatabaseSetup(
    session=session,
    database=config.snowflake.database,
    schema_name=config.snowflake.schema_name,
    warehouse=config.snowflake.warehouse,
)

db_result = db_setup.run(warehouse_size="MEDIUM")
print(f"\nDatabase setup complete: {db_result}")

## Create Tables

Create 5 tables to support the ML pipeline:

| Table | Purpose |
|-------|---------|
| `RAW_PATIENT_DATA` | Historical training data |
| `STREAMING_PATIENT_DATA` | Real-time incoming data |
| `MODEL_METRICS` | Evaluation metrics history |
| `BASELINE_PATIENT_DATA` | Baseline for drift detection |
| `TEST_PATIENT_DATA` | Held-out test set |

> **Note:** Inference results are automatically captured via Snowflake's autocapture feature and accessed through `INFERENCE_TABLE()`.

In [ ]:
from setup.tables_setup import TablesSetup

tables_setup = TablesSetup(
    session=session,
    database=config.snowflake.database,
    schema_name=config.snowflake.schema_name,
)

tables_result = tables_setup.run(warehouse=config.snowflake.warehouse)
print(f"\nTables created: {tables_result['tables']}")

## Create Stages

Create internal stages for file storage:

| Stage | Purpose |
|-------|---------|
| `MODEL_ARTIFACTS` | Serialized models (pickle files) |
| `JOB_PAYLOADS` | SPCS job scripts and configs |
| `DATA_STAGE` | Intermediate data files |
| `EVALUATION_RESULTS` | Evaluation reports and metrics |

In [ ]:
from setup.stages_setup import StagesSetup

stages_setup = StagesSetup(
    session=session,
    database=config.snowflake.database,
    schema_name=config.snowflake.schema_name,
)

stages_result = stages_setup.run()
print(f"\nStages created: {stages_result['stages']}")

## Create Compute Pool

Create a compute pool for ML Jobs (SPCS workloads):

| Setting | Value | Purpose |
|---------|-------|---------|
| `INSTANCE_FAMILY` | CPU_X64_S | Small CPU instances for training |
| `MIN_NODES` | 1 | Minimum nodes to keep warm |
| `MAX_NODES` | 3 | Maximum nodes for scaling |
| `AUTO_SUSPEND_SECS` | 300 | Suspend after 5 minutes idle |

In [ ]:
from setup.compute_pool_setup import ComputePoolSetup

compute_pool_setup = ComputePoolSetup(
    session=session,
    compute_pool_name=config.compute.compute_pool,
    instance_family=config.compute.instance_family,
    min_nodes=config.compute.min_nodes,
    max_nodes=config.compute.max_nodes,
)

compute_pool_result = compute_pool_setup.run(resume=True)
print(f"\nCompute pool setup complete:")
print(f"  Name: {compute_pool_result['compute_pool']}")
print(f"  Instance Family: {compute_pool_result['instance_family']}")
print(f"  Status: {compute_pool_result['status']}")

## Verify Setup

In [ ]:
print("Tables in schema:")
session.sql(f"SHOW TABLES IN SCHEMA {config.full_schema}").show()

print("\nStages in schema:")
session.sql(f"SHOW STAGES IN SCHEMA {config.full_schema}").show()

---
## Summary

Setup complete! You have created:
- Database and schema
- Warehouse for compute
- 6 tables for the ML pipeline
- 4 internal stages for artifacts

**Next:** Continue to `01-Data-Generation.ipynb` to generate synthetic patient data.